# 1. Load data

This notebook produces results for the RF, ET, XGB, and LGBM results. Tune period and excess returns in this cell. Results are output in `tree_runs.csv`, which we manually clean up for clearer reporting.

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import seaborn as sns
import utils.base_utils as bu
import utils.window_utils as wu
import numpy as np
import matplotlib.pyplot as plt
from utils.publication_lags import apply_fred_md_publication_lag
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

repo_root = os.path.abspath('../..')

# Bianchi period:
start_date = '1971-08-31'
# end_date = '2018-12-31'
end_date = '2025-06-30' # kr and gsw end date
DATASET = 'lw' # or 'kr
GAP = 11
REVISED = False # Tune manually, True if use FRED, False if use ALFRED.
MONTHLY = False

if MONTHLY:
    USE_MONTHLY_FORWARDS = True
else: 
    USE_MONTHLY_FORWARDS = False

maturities = [str(i) for i in range(12, 121) if i % 12 == 0] # select only yearly maturities

yields = bu.get_yields(type=DATASET, start=start_date, end=end_date, maturities=maturities) # type can be kr, lw, gsw
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna() # horizon=12 means holding for 12 months

# adjust fred_md start_date by 6 months to fetch enough data for shifting
fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv', start=fred_md_start_date, end=end_date) # this is aligned to the last day of the previous month, so we get the same number of observations as the yields data

monthly_yields = bu.get_yields(type=DATASET, start=start_date, end=end_date, maturities=[str(i) for i in range(1, 121)]) # needed for monthly holding period excess returns. Not available for gsw
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna() # calculate monthly excess returns for robustness
monthly_forward = bu.get_monthly_forward_rates(monthly_yields)
# If wanted, apply per-series publication lag to latest-snapshot macro data
# from utils.publication_lags import apply_fred_md_publication_lag
# fred_md = apply_fred_md_publication_lag(fred_md_raw)  
# For results in paper, we naively shift all FRED-MD series by 1 month
# to reflect publication lag:
fred_md = fred_md_raw.shift(1)

# Drop TWEXAFEGSMTHx and ACOGNO as they start late
fred_md = fred_md.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'])
# Finally, revert fred_md to start_date, after transformations and lag adjustments
fred_md = fred_md[start_date:end_date]

# Drop dates outside the xr range
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]
monthly_forward = monthly_forward.loc[monthly_forward.index <= xr.index[-1]]

# Construct X with 3-level MultiIndex: (source, group, series)
s2g = build_full_group_mapping(fred_md, forward, yields)

if USE_MONTHLY_FORWARDS:
    forward = monthly_forward.copy()

X = pd.concat([fred_md, forward, yields],
               axis=1,
               keys=['fred', 'forward', 'yields'])

X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')
# For now, forward fill X
X = X.ffill()

groups = groups_as_array(X, level='group')
MATURITIES = ['24','36','48','60','84','120']

if MONTHLY:
    y_all = monthly_xr[MATURITIES].values
else:
    y_all = xr[MATURITIES].values

dates = xr.index

/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:219: UserWarning: The following series are defined in get_fredmd_grouping() but are not present in the FRED-MD data: ['ACOGNO', 'TWEXAFEGSMTHx']. They may have been dropped or renamed.
  warnings.warn(
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(


# 2. Run models

In [5]:
from models.base import *
from models.classical import *
from models.gbt import *
from models.linear import *
from models.tree import *
from tqdm import tqdm
from datetime import datetime
from joblib import Parallel, delayed

# y = monthly_xr['120'].values # 1-month excess returns
# y = xr['120'].values # 10-year overlapping excess returns
OOS_start_1 = pd.Timestamp('1990-01-31')
OOS_start_2 = pd.Timestamp('2000-01-31')

coefs = []

def record_coef(coef):
    coefs.append(np.abs(coef).copy())  # store absolute value

tree_models = {
    # "Tree_RF": FwdFredTreeEnsemble1D(estimator="rf"),
    # "Tree_ET": FwdFredTreeEnsemble1D(estimator="ef"),
    "Tree_XGB": FwdFredTreeEnsemble1D(estimator="xgb"),
    # "Tree_LGBM": FwdFredTreeEnsemble1D(estimator="lgbm"),

    # Forward-only variants:
    # "Tree_RF_FwdOnly": FwdFredTreeEnsemble1D(estimator="rf", include_fred=False),
    # "Tree_ET_FwdOnly": FwdFredTreeEnsemble1D(estimator="ef", include_fred=False),
    "Tree_XGB_FwdOnly": FwdFredTreeEnsemble1D(estimator="xgb", include_fred=False),
    # "Tree_LGBM_FwdOnly": FwdFredTreeEnsemble1D(estimator="lgbm", include_fred=False)
}

all_models = {**tree_models}

results = {}

# Helper function for evaluation of a single model+maturity combination
def evaluate_model_maturity(name, model, maturity, X, y_dict, dates, OOS_start_1, OOS_start_2):
    """
    Evaluate a single model on a single maturity in isolation.
    Returns a tuple: (model_name, maturity, r2_1, signif_1, r2_2, signif_2)
    """
    try:
        y = y_dict[maturity]
        y_forecast = wu.expanding_window(
            model, X, y, dates, OOS_start_1, 
            gap=GAP,         # gap = 11 for annual nonoverlapping yields
            refit_freq=1,  # Refit every month
            realtime=not REVISED
        )
        
        # R2 for OOS_start_1
        r2_1 = wu.oos_r2(y, y_forecast, gap=GAP)
        signif_1 = bu.RSZ_Signif(y, y_forecast, gap=GAP)
        
        # R2 for OOS_start_2
        y_forecast_2 = y_forecast.copy()
        y_forecast_2[dates < OOS_start_2] = np.nan
        r2_2 = wu.oos_r2(y, y_forecast_2, gap=GAP)
        signif_2 = bu.RSZ_Signif(y, y_forecast_2, gap=GAP)
        
        return (name, maturity, r2_1, signif_1, r2_2, signif_2)
    except Exception as e:
        print(f"Error evaluating {name} on maturity {maturity}: {e}")
        return (name, maturity, np.nan, np.nan, np.nan, np.nan)

# Prepare y_dict for execution
if MONTHLY:
    y_dict = {mat: monthly_xr[mat].values for mat in ['24','36','48','60','84','120']}
else:
    y_dict = {mat: xr[mat].values for mat in ['24','36','48','60','84','120']}

# Create list of (model_name, model_instance, maturity) tuples
task_list = [
    (name, model, maturity)
    for name, model in all_models.items()
    for maturity in ['24','36','48','60','84','120']
    # for maturity in ['120']
]

print(f"Running {len(task_list)} model-maturity combinations sequentially...")

csv_path = os.path.join(repo_root, 'notebooks', 'orchestrators', 'results', 'tree_results.csv')

print("\n=== RESULTS ===")
# Execute in parallel using joblib
results_list = Parallel(n_jobs=6)(
    delayed(evaluate_model_maturity)(name, model, maturity, X, y_dict, dates, OOS_start_1, OOS_start_2)
    for name, model, maturity in tqdm(task_list, desc="Parallel Evaluation")
)

# Append after all tasks complete
for res in results_list:
    name, maturity, r2_1, signif_1, r2_2, signif_2 = res
    print(f"Model: {name}\tMaturity {maturity}:\n R2_OOS (1990) = {r2_1:.4f}\n Significance (1990): {signif_1:.3f}\n R2_OOS (2000) = {r2_2:.4f}\n Significance (2000): {signif_2:.3f}\n")
    
    # Save partial results immediately
    results_df = pd.DataFrame([{
        'time_now': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'name': name,
        'maturity': maturity,
        'signif_1990': signif_1,
        'r2_1990': r2_1,
        'signif_2000': signif_2,
        'r2_2000': r2_2,
        'start_date': start_date,
        'end_date': end_date,
        'yield_type': f'{DATASET}',
        'f"Revised macro data: {revised}"': f'Revised macro data: {REVISED}',
        'f"gap={gap}"': f'gap={GAP}'
    }])
    
    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path)
        existing_df.columns = existing_df.columns.str.strip()
        existing_df = existing_df.loc[:, ~existing_df.columns.duplicated()]
        results_df = pd.concat([existing_df, results_df], ignore_index=True)

    results_df.to_csv(csv_path, index=False)

print(f"\nAll results saved to {csv_path}")

Running 12 model-maturity combinations sequentially...

=== RESULTS ===


Parallel Evaluation: 100%|██████████| 12/12 [14:10<00:00, 70.84s/it] 


Model: Tree_XGB	Maturity 24:
 R2_OOS (1990) = -0.1267
 Significance (1990): 0.076
 R2_OOS (2000) = -0.2237
 Significance (2000): 0.291

Model: Tree_XGB	Maturity 36:
 R2_OOS (1990) = -0.1043
 Significance (1990): 0.072
 R2_OOS (2000) = -0.2110
 Significance (2000): 0.253

Model: Tree_XGB	Maturity 48:
 R2_OOS (1990) = -0.2167
 Significance (1990): 0.172
 R2_OOS (2000) = -0.3119
 Significance (2000): 0.330

Model: Tree_XGB	Maturity 60:
 R2_OOS (1990) = -0.1710
 Significance (1990): 0.104
 R2_OOS (2000) = -0.2773
 Significance (2000): 0.245

Model: Tree_XGB	Maturity 84:
 R2_OOS (1990) = -0.1647
 Significance (1990): 0.119
 R2_OOS (2000) = -0.3075
 Significance (2000): 0.327

Model: Tree_XGB	Maturity 120:
 R2_OOS (1990) = -0.2112
 Significance (1990): 0.167
 R2_OOS (2000) = -0.2471
 Significance (2000): 0.195

Model: Tree_XGB_FwdOnly	Maturity 24:
 R2_OOS (1990) = -0.4493
 Significance (1990): 0.726
 R2_OOS (2000) = -0.3992
 Significance (2000): 0.718

Model: Tree_XGB_FwdOnly	Maturity 36:
 R